# Fine-tuning QLoRA — Text2Cypher em portugues (fluxo assistencial do SUS)

Ajusta um modelo pequeno e aberto (Qwen2.5-1.5B-Instruct) com **QLoRA** para traduzir
perguntas em portugues para consultas Cypher sobre o grafo do SUS. Roda na **GPU gratuita
do Kaggle** (T4). O dataset vem da destilacao do Gemini (`src/gerar_dataset.py`).

## Antes de rodar
1. Kaggle: **Settings -> Accelerator -> GPU T4 x1**.
2. Faca upload do `dataset_text2cypher.jsonl` (gerado por `src/gerar_dataset.py`) como
   **Kaggle Dataset** e adicione ao notebook. Ajuste `DATASET_PATH` na celula de config.
3. Rode as celulas em ordem. Ao final, baixe a pasta `adapter/` (o adapter LoRA) e
   aponte `ADAPTER_DIR` no `.env` do repo para servir o modelo localmente.

O objetivo e comprovar o fluxo LoRA/QLoRA: destilar do professor -> treinar o aluno ->
avaliar com a mesma metrica do `src/avaliar.py` -> comparar com a API.

## 1. Dependencias (instala no ambiente do Kaggle)

In [ ]:
!pip install -q -U transformers==4.44.2 peft==0.13.2 bitsandbytes==0.44.1 \
    trl==0.11.4 datasets==3.0.1 accelerate==0.34.2

## 2. Configuracao

In [ ]:
import os, json, re, glob, torch
from pathlib import Path

BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"   # cabe na T4 em 4-bit

# auto-descobre o dataset onde quer que o Kaggle o monte (robusto ao nome da pasta)
if os.path.exists("/kaggle/input"):
    print("inputs montados:", os.listdir("/kaggle/input"))
_cand = glob.glob("/kaggle/input/**/dataset_text2cypher.jsonl", recursive=True)
DATASET_PATH = _cand[0] if _cand else "/kaggle/input/text2cypher-sus/dataset_text2cypher.jsonl"
print("DATASET_PATH:", DATASET_PATH, "| existe:", os.path.exists(DATASET_PATH))
OUT_DIR      = "/kaggle/working/adapter"
MAX_LEN      = 1024
VAL_FRAC     = 0.1
SEED         = 42

ESQUEMA = """Grafo Neo4j do fluxo assistencial do SUS (recorte oncologico, Sao Paulo).
Nos: (:RegiaoSaude {codigo,nome,drs,polo_oncologico}); (:Municipio {codigo,nome,populacao,idhm_renda,dist_polo_km}); (:Estabelecimento {cnes,nome,tipo}); (:Procedimento {codigo,nome,grupo,linha,complexidade,cid_grupo}).
Relacoes: (:Municipio)-[:PERTENCE_A]->(:RegiaoSaude); (:RegiaoSaude)-[:FLUXO {ano,complexidade,linha,volume,deslocou}]->(:RegiaoSaude) (origem=residencia, destino=atendimento, deslocou=true se origem<>destino); (:Estabelecimento)-[:LOCALIZADO_EM]->(:Municipio); (:Estabelecimento)-[:REALIZA {ano,volume}]->(:Procedimento).
complexidade em {basica,media,alta}; linha em {radioterapia,quimioterapia,diagnostico}."""

SYSTEM = (
    "Voce traduz perguntas em portugues para consultas Cypher sobre um grafo Neo4j. "
    "Use SOMENTE os rotulos, relacoes e propriedades do esquema. Responda com UMA "
    "consulta Cypher, sem explicacao.\n\nESQUEMA:\n" + ESQUEMA
)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## 3. Dataset -> formato de chat (train/val)

In [ ]:
from datasets import Dataset

registros = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if linha:
            registros.append(json.loads(linha))
print("pares carregados:", len(registros))

def to_messages(r):
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": r["pergunta"]},
        {"role": "assistant", "content": r["cypher"]},
    ]}

ds = Dataset.from_list([to_messages(r) for r in registros]).shuffle(seed=SEED)
n_val = max(1, int(len(ds) * VAL_FRAC))
val_ds, train_ds = ds.select(range(n_val)), ds.select(range(n_val, len(ds)))
print("treino:", len(train_ds), "| val:", len(val_ds))

## 4. Modelo base em 4-bit (quantizacao QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16
)
model.config.use_cache = False

## 5. Adapter LoRA

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Treino (SFT sobre a resposta)

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir="/kaggle/working/ckpt",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=MAX_LEN,
    packing=False,
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, processing_class=tok)
trainer.train()

## 7. Salva o adapter

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("adapter salvo em", OUT_DIR, "-> baixe esta pasta e aponte ADAPTER_DIR no .env")

## 8. Avaliacao rapida no conjunto de validacao

Mesma logica do `src/avaliar.py`: normaliza e compara o Cypher gerado com a referencia
(acerto exato normalizado). A validacao de execucao contra o Neo4j fica no `src/avaliar.py`,
com o grafo carregado. Aqui medimos o quanto o aluno reproduz o professor.

In [ ]:
def normaliza(c):
    c = re.sub(r"\s+", " ", c.strip().lower())
    return c.rstrip(";").strip()

def gera(pergunta, max_new=160):
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":pergunta}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return txt.strip().strip("`").replace("cypher\n", "").strip()

acertos = 0
for ex in val_ds:
    perg = ex["messages"][1]["content"]
    ref  = ex["messages"][2]["content"]
    pred = gera(perg)
    ok = normaliza(pred) == normaliza(ref)
    acertos += ok
    print(("OK " if ok else "-- "), perg[:60])
    if not ok:
        print("    ref :", ref[:90])
        print("    pred:", pred[:90])
print(f"\nAcerto exato normalizado: {acertos}/{len(val_ds)} = {acertos/len(val_ds):.1%}")